In [1]:
import numpy as np
import pandas as pd

In [11]:
samples = [
    "20221128_K562_Actd_0h_rep1",
    "20221128_K562_Actd_0h_rep2",
    "20221128_K562_Actd_3h_rep1",
    "20221128_K562_Actd_3h_rep2",
    "20221128_K562_Actd_6h_rep1",
    "20221128_K562_Actd_6h_rep2",
]

m = pd.DataFrame(index=pd.Index(samples, name="Sample"))

vs1, vs2 = [], []
for s in samples:
    for line in open("results/01_prepare/01_cutadapt/%s.log" % s):
        if "Total read pairs processed:" in line:
            vs1.append(int(line.strip().split()[-1].replace(",", "")))
        if "Pairs written (passing filters):" in line:
            vs2.append(int(line.strip().split()[-2].replace(",", "")))
m["Reads"] = vs1
m["Trimmed.Reads"] = vs2
m["Trimmed.Reads.Ratio"] = m["Trimmed.Reads"] / m["Reads"]

vs = []
for s in samples:
    for line in open("results/02_mapping/02_star_mapping/%s/%s.Log.final.out" % (s, s)):
        if "input reads" in line:
            vs.append(int(line.strip().split()[-1]))
m["Clean.Reads"] = vs
m["Ribo.Reads"] = m["Trimmed.Reads"] - m["Clean.Reads"]
m["Ribo.Reads.Ratio"] = m["Ribo.Reads"] / m["Trimmed.Reads"]

vs1, vs2 = [], []
for s in samples:
    for line in open("results/02_mapping/02_star_mapping/%s/%s.Log.final.out" % (s, s)):
        if "Uniquely mapped reads number" in line:
            vs1.append(int(line.strip().split()[-1]))
        if "Number of reads mapped to multiple loci" in line:
            vs2.append(int(line.strip().split()[-1]))
m["Uniq.Reads"] = vs1
m["Uniq.Reads.Ratio"] = m["Uniq.Reads"]/m["Clean.Reads"]
m["Multi.Reads"] = vs2
m["Multi.Reads.Ratio"] = m["Multi.Reads"] / m["Clean.Reads"]

vs = []
for s in samples:
    for line in open("results/02_mapping/03_bam_filtered/%s.human.flagstat" % s):
        if "read1" in line:
            vs.append(int(line.strip().split()[0]))
m["Human.Reads"] = vs

vs = []
for s in samples:
    for line in open("results/02_mapping/03_bam_filtered/%s.fly.flagstat" % s):
        if "read1" in line:
            vs.append(int(line.strip().split()[0]))
m["Fly.Reads"] = vs

m["Filtered.Reads"] = m["Human.Reads"] + m["Fly.Reads"]
m["Human.Reads.Ratio"] = m["Human.Reads"] / m["Filtered.Reads"]
m["Fly.Reads.Ratio"] = m["Fly.Reads"] / m["Filtered.Reads"]

m.to_csv("reports/rnaseq_actd_summary.csv")
m

,Reads,Trimmed.Reads,Trimmed.Reads.Ratio,Clean.Reads,Ribo.Reads,Ribo.Reads.Ratio,Uniq.Reads,Uniq.Reads.Ratio,Multi.Reads,Multi.Reads.Ratio,Human.Reads,Fly.Reads,Filtered.Reads,Human.Reads.Ratio,Fly.Reads.Ratio
Sample,,,,,,,,,,,,,,,
20221128_K562_Actd_0h_rep1,32752869,32467300,0.991281,32139534,327766,0.010095,30707880,0.955455,1074216,0.033424,26574178,2596097,29170275,0.911002,0.088998
20221128_K562_Actd_0h_rep2,34770161,34391828,0.989119,34049416,342412,0.009956,32565284,0.956412,1123581,0.032999,28237220,2744271,30981491,0.911422,0.088578
20221128_K562_Actd_3h_rep1,43632098,42584166,0.975983,41441716,1142450,0.026828,39445861,0.951839,1544119,0.037260,33576584,4288480,37865064,0.886743,0.113257
20221128_K562_Actd_3h_rep2,37574447,37038292,0.985731,36047274,991018,0.026757,34293668,0.951353,1324246,0.036736,29250768,3691389,32942157,0.887943,0.112057
20221128_K562_Actd_6h_rep1,40107187,38408565,0.957648,37272475,1136090,0.029579,35350810,0.948443,1443879,0.038738,29234661,4863303,34097964,0.857373,0.142627
20221128_K562_Actd_6h_rep2,45673485,43830856,0.959656,42526251,1304605,0.029765,40276126,0.947089,1685157,0.039626,33291317,5482438,38773755,0.858604,0.141396
